<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style='height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0;'/>
    <div style='border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;'>
        <h1 style='margin: 0 0 8px 0; color: white; font-size: 1.5em;'>Ingeniería y Ciencia de Datos</h1>
        <h3 style='margin: 0 0 8px 0; color: white; font-size: 1.5em;'>Laboratorio de Procesamiento de Datos</h3>
        <h3 style='margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;'>Módulo 2: Codificación de Variables Categóricas Ordinales</h3>
    </div>
</div>


Una variable categórica **ordinal** tiene categorías con un orden definido, pero no necesariamente con distancias iguales entre ellas. Por ejemplo, `Primaria < Secundaria < Preparatoria < Universidad` expresa un orden, pero no implica que la diferencia entre cada nivel sea constante.

En este notebook compararemos una asignación ordinal simple con contrastes polinomiales. 

## Codificación ordinal simple

Asigna un entero a cada categoría siguiendo un orden definido por el problema. Por ejemplo: `Primaria = 1`, `Secundaria = 2`, `Preparatoria = 3` y `Universidad = 4`.

Este código es sencillo y conserva el orden, pero muchos modelos interpretan las diferencias numéricas como distancias. Aquí se asume que avanzar de un nivel al siguiente tiene el mismo efecto, supuesto que debe justificarse. 

In [ ]:
# Ejemplo de codificación ordinal simple en pandas
import pandas as pd

df = pd.DataFrame({
    'Nivel de Educación': ['Primaria', 'Secundaria', 'Universidad', 'Preparatoria', 'Secundaria']
})

In [ ]:
orden = ['Primaria', 'Secundaria', 'Preparatoria', 'Universidad']
df['Nivel Codificado'] = df['Nivel de Educación'].apply(lambda x: orden.index(x) + 1)
df

## Contrastes polinomiales

Los contrastes polinomiales representan patrones entre niveles ordenados mediante componentes lineales, cuadráticos y de mayor grado. El componente lineal captura una tendencia creciente o decreciente, el cuadrático puede capturar una forma curva. Cada nivel recibe valores numéricos calculados a partir del orden elegido.

En las celdas siguientes construiremos esos valores paso a paso y observaremos cómo se relacionan con una variable numérica mediante una regresión lineal. La elección del orden es parte del modelo, no un detalle de implementación. En particular, los días de la semana son cíclicos, no una escala lineal: su ejemplo sirve para practicar la construcción de contrastes, no para afirmar que lunes y domingo están a una distancia lineal.

In [ ]:
import pandas as pd
import numpy as np

data = pd.read_csv('Data/temps.txt', sep=r'\s+', header=None)
data.columns = ['Temperatura máxima', 'Temperatura mínima', 'HDD', 'CDD', 'Rain']
d = ['Sábado', 'Domingo', 'Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes']
dias=[d[i%7] for i in range(len(data))]
catego=['Soleado','Mayormente soleado','Parcialmente nublado','Mayormente nublado','Nublado','Lluvioso']
prono=[catego[-1] if data['Rain'][i]>0 else catego[int(round(1.2*(data['HDD'][i]-data['CDD'][i])+2))] for i in range(len(data))]
df=pd.DataFrame(columns=['Días','Pronóstico','Temperatura máxima','Temperatura mínima'],index=range(len(data)))
df.iloc[:,2:4]=data.iloc[:,0:2]
df['Días']=dias
df['Pronóstico']=prono
df.head()

### Componente lineal

Asignaremos a cada nivel un valor centrado alrededor de cero. El orden de los niveles define qué categoría recibe cada valor, mantener ese orden explícito evita depender de cómo aparecieron las categorías en los datos.

Primero contamos los niveles ($n$) y construimos puntuaciones centradas $x$. Para siete niveles, por ejemplo, una secuencia posible es $-3,-2,-1,0,1,2,3$. Cada puntuación se asigna a una categoría respetando el orden definido.

Estos valores son **códigos**, no mediciones de distancia entre categorías. El contraste lineal permite probar una tendencia general, la escala concreta puede cambiar, pero no debe cambiar el orden. La variable `x` del código se reutiliza más adelante para construir el componente cuadrático.

In [ ]:
data

In [ ]:
df['Días'].nunique()

In [ ]:
#Codificación para la variable `Días`
n=len(df['Días'].unique())
lim=n-1 if n%2==0 else (n-1)//2
step = 2 if n%2==0 else 1
x=range(-lim,lim+1,step)
list(x)

In [ ]:
catego=df['Días'].unique()
catego

In [ ]:
Días_ord = [
    día for día in ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
    if día in df['Días'].unique()
]
Días_ord

In [ ]:
a0=0
a1=5 #-0.1
Días_lin={categorías:a1*i+a0 for categorías,i in zip(Días_ord,x)}
Días_lin

In [ ]:
df['Días_lin']=df['Días'].map(Días_lin)
df.head()

In [ ]:
df['Pronóstico'].nunique()

In [ ]:
#Codificando la variable `Pronóstico`
n=len(df['Pronóstico'].unique())
lim=n-1 if n%2==0 else (n-1)//2
step=2 if n%2==0 else 1
x=range(-lim,lim+1,step)
list(x)

In [ ]:
cat=df['Pronóstico'].unique()
cat

In [ ]:
Pronóstico_ord = [
    categoría for categoría in [
        'Soleado', 'Mayormente soleado', 'Parcialmente nublado',
        'Mayormente nublado', 'Nublado', 'Lluvioso'
    ]
    if categoría in df['Pronóstico'].unique()
]
Pronóstico_ord

In [ ]:
a0=0
a1=-1
Pronóstico_lin={categorías:a1*i+a0 for categorías,i in zip(Pronóstico_ord,x)}
Pronóstico_lin

In [ ]:
df['Pronóstico_lin']=df['Pronóstico'].map(Pronóstico_lin)
df.head()

## Componente cuadrático

A partir de las mismas puntuaciones lineales $x$, construimos un segundo contraste usando $x^2$ y centramos sus valores para que sumen cero. En una regresión, incluir ambos componentes permite describir una tendencia y una curvatura.

El contraste cuadrático no significa automáticamente que la relación sea cóncava: el signo de su coeficiente en el modelo determina la dirección. 

In [ ]:
#Codificación de la variable `Pronóstico`
a2=-1
a1=0
a0=-a2/n*np.sum([i**2 for i in x])
Pronóstico_cuad={categorías:a0+a1*i+a2*i**2 for categorías,i in zip(Pronóstico_ord,x)}
Pronóstico_cuad

In [ ]:
np.sum(list(Pronóstico_lin.values()))

In [ ]:
df['Pronóstico_cuad']=df['Pronóstico'].map(Pronóstico_cuad)
df.head()

In [ ]:
#Codificación de `Días`
n=len(df['Días'].unique())
lim=n-1 if n%2==0 else (n-1)//2
step=2 if n%2==0 else 1
x=range(-lim,lim+1,step)
a2=1
a1=0
a0=-a2/n*np.sum([i**2 for i in x])
Días_cuad={categos:a0+a1*i+a2*i**2 for categos,i in zip(Días_ord,x)}
Días_cuad

In [ ]:
np.sum(list(Días_cuad.values()))

In [ ]:
df['Días_cuad']=df['Días'].map(Días_cuad)
df.head()

## Relación con la temperatura mínima

Ajustaremos una regresión con las componentes lineal y cuadrática del pronóstico para describir su asociación con la temperatura mínima. Los diagramas comparan los valores observados con las predicciones del modelo; son una demostración del efecto de la codificación, no una evaluación de calidad predictiva.

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
Pronóstico_lin


In [ ]:
df['Pronóstico_lin']

In [ ]:
plt.scatter(df['Pronóstico_lin'],df['Temperatura mínima'])
plt.xlabel('Pronóstico_lin')
plt.ylabel('Temperatura mínima')
plt.grid()

In [ ]:
Pronóstico_cuad

In [ ]:
plt.scatter(df['Pronóstico_cuad'],df['Temperatura mínima'])
plt.grid()

In [ ]:
df['Pronóstico_cuad']

In [ ]:
lin=LinearRegression()
lin.fit(df[['Pronóstico_lin','Pronóstico_cuad']],df['Temperatura mínima'])

In [ ]:
f=lin.predict(df[['Pronóstico_lin','Pronóstico_cuad']])
f

In [ ]:
f=lin.predict(df[['Pronóstico_lin','Pronóstico_cuad']])
plt.scatter(df['Pronóstico_lin'],df['Temperatura mínima'],label='Observaciones')
plt.scatter(df['Pronóstico_lin'],f,label='Predicción')
plt.xlabel('Pronóstico_lin')
plt.ylabel('Temperatura mínima')
plt.legend()
plt.grid()

In [ ]:
plt.scatter(df['Pronóstico_cuad'],df['Temperatura mínima'],label='Observaciones')
plt.scatter(df['Pronóstico_cuad'],f,label='Predicción')
plt.legend()
plt.xlabel('Pronóstico_cuad')
plt.ylabel('Temperatura mínima')
plt.grid()

### Asignación creciente

Desplazar toda la secuencia por una constante cambia los valores numéricos, pero conserva el orden. El desplazamiento por sí solo no añade información ni mejora la codificación.

escogiendo $$ X={0,1,2,3,...} $$

In [ ]:
n=len(df['Días'].unique())
lim=n-1 if n%2==0 else (n-1)//2
step = 2 if n%2==0 else 1
x=range(-lim,lim+1,step)
x_list = list(x)
x_list = [x_list[i]+3 for i in range(len(x_list))]
x_list

In [ ]:
catego=df['Días'].unique()
catego

In [ ]:
Días_ord=np.concatenate([catego[2:],catego[:2]])
Días_ord

In [ ]:
a0=0
a1=5 #-0.1
Días_crec={categorías:a1*i+a0 for categorías,i in zip(Días_ord,x_list)}
Días_crec

In [ ]:
df['Días'].map(Días_crec)

### Concava con a2 positivo

$$n*a0+\sum(a1*x+a2*x**2)=0$$

In [ ]:
a2=1
a1=0
a0=-a2/n*np.sum([i**2 for i in x])
Pronóstico_cuad_con={categorías:a0+a1*i+a2*i**2 for categorías,i in zip(Pronóstico_ord,x)}
Pronóstico_cuad_con

## Guía rápida para elegir una codificación

| Método | Representación | Qué patrón expresa | Supuesto o precaución |
|---|---|---|---|
| Ordinal simple | Un entero por nivel (`1, 2, ..., K`) | Conserva el orden de las categorías | En muchos modelos, las diferencias entre enteros se interpretan como distancias |
| Contraste lineal | Una puntuación centrada por nivel | Tendencia creciente o decreciente | El orden y la dirección deben definirse antes de codificar |
| Contraste cuadrático | Una segunda columna construida con $x^2$ | Curvatura entre niveles | No impone por sí solo que la respuesta suba o baje de forma monótona |
| Puntuación creciente | Transformación afín `a0 + a1*x`, con `a1 > 0` | Mantiene el orden ascendente | Cambiar origen o escala cambia los coeficientes, no el orden |
| Puntuación cóncava/convexa | Polinomio con término `a2*x^2` | Curvatura hacia abajo (`a2 < 0`) o arriba (`a2 > 0`) | La forma debe justificarse con el problema; no se deduce del orden solamente |

**Importante:** los contrastes polinomiales son útiles cuando el orden tiene sentido y se desea resumir una tendencia. Si no se quiere imponer una forma concreta, una codificación indicadora permite estimar un efecto distinto para cada nivel.

## Práctica de Laboratorio: Niveles de habitaciones en Melbourne

Usaremos `Data/melb_data.csv`. La variable original `Rooms` es un conteo numérico; para practicar codificación ordinal, la agruparemos en niveles ordenados: `1-2`, `3-4` y `5+` habitaciones.

1. Revisa cuántas observaciones hay en cada nivel y el precio mediano asociado.
2. Asigna códigos ordinales `1`, `2` y `3` en el orden definido.
3. Compara el número de columnas de la codificación ordinal y one-hot.
4. ¿Qué supuesto de distancia hace un modelo lineal cuando usa los códigos 1, 2 y 3?
5. Compara esa interpretación con one-hot, que permite un efecto independiente por nivel.


In [ ]:
df = pd.read_csv('Data/melb_data.csv')
df